# The Vibe Tax — the extractor & grader, explained step by step

A **glass-box** notebook: pick one problem, and every stage runs in its own cell showing
its intermediate value — how the raw model reply becomes runnable code, and how one test's
**input/output** is checked. Built for walking through with an advisor.

Standalone: no git clone. Upload `lcb_v3_responses.json` (required); `lcb_problems.jsonl`
(optional, for the clean statement) and `lcb_tests.jsonl` (optional, for the grading section).

## 0 · Pick a problem
Set the task and (optionally) which framing/model and which test to walk through. Re-run
from here to switch problems.

In [ ]:
TASK_ID    = "lcb/3517"          # the LCB problem to trace
CONDITION  = "agentic_terse"     # agentic_terse | agentic_casual | webchat_detailed | webchat_multilingual
MODEL      = "chatgpt"           # chatgpt | claude | codestral
TEST_INDEX = 0                   # which test case to walk through in the grading section
print("tracing:", TASK_ID, "|", CONDITION, "|", MODEL, "| test #", TEST_INDEX)

## 1 · Upload the data

In [ ]:
import json
try:
    from google.colab import files
    up = files.upload()
except Exception:
    up = {n: open(n, "rb").read() for n in
          ("lcb_v3_responses.json", "lcb_problems.jsonl", "lcb_tests.jsonl") if __import__("os").path.exists(n)}
PROBS, RESP, TESTS = {}, {}, {}
for name, data in up.items():
    text = data.decode("utf-8", errors="replace")
    if text.lstrip().startswith("["):
        RESP = {(r["task_id"], r["level"], r["model"]): r for r in json.loads(text)}
        print(f"[responses] {len(RESP)}")
    else:
        rows = [json.loads(l) for l in text.splitlines() if l.strip()]
        if rows and ("public_tests" in rows[0] or "private_tests" in rows[0]):
            TESTS = {d["task_id"]: d for d in rows}; print(f"[tests] {len(TESTS)}")
        else:
            PROBS = {d["task_id"]: d for d in rows}; print(f"[problems] {len(PROBS)}")
assert RESP, "lcb_v3_responses.json is required."

## 2 · Define the primitives
The exact functions from `score_lcb.py`, plus the regexes the extractor uses (exposed as
named constants so the step cells below can show each one).

In [ ]:
# --- core primitives, copied verbatim from score_lcb.py (standalone) ---
import ast, json, re, time, multiprocessing, inspect

_IMPORTS = 'from typing import *\nimport collections, math, heapq, bisect, itertools, functools, re\nfrom collections import *\nfrom math import *\nfrom functools import *\n'
PER_TEST_TIMEOUT = 6
PRIVATE_ONLY = False
FENCE_RE = '```(?:python|py)?\\s*\\n(.*?)```'
CONSTRUCT_RE = '(?m)^(?:class|def|import|from|@)\\s'
CODEISH_RE = '(?m)^\\s*(?:class|def|import|from|@)\\s'

def parse_lit(s):
    if not isinstance(s, str):
        return s
    s = s.strip()
    for fn in (json.loads, ast.literal_eval):
        try:
            return fn(s)
        except Exception:
            continue
    return s  # leave as raw string


def parse_input(inp):
    """LCB functional input = one literal per line -> list of args."""
    if isinstance(inp, list):
        return [parse_lit(x) for x in inp]
    return [parse_lit(line) for line in str(inp).split("\n") if line.strip() != ""]


def _trim_to_compilable(code):
    """Drop trailing lines until the source parses. Conversational replies put a
    prose paragraph AFTER the code ('This checks every adjacent pair...'); with no
    fences to delimit it, that prose used to be exec'd and threw SyntaxError,
    failing correct code. The polite/detailed framing elicits more explain-after-
    code, so this bug penalized it hardest — a scoring artifact, not correctness.
    We keep the largest leading prefix that is valid Python and still has the
    class/method."""
    lines = code.split("\n")
    while lines:
        src = "\n".join(lines)
        try:
            ast.parse(src)
            return src
        except SyntaxError:
            lines.pop()
    return None


def extract_solution(completion, entry):
    """Pull runnable code that defines `class Solution` (or the method), robust to
    trailing prose, leading chatter, AND helper definitions (Fenwick/DSU/Segments…)
    the model defines BEFORE `class Solution`. Returns the first candidate that
    compiles and still defines the target. Candidates that KEEP the helpers are
    tried first, so we don't hand exec() a Solution that references an undefined
    helper we chopped off."""
    if not completion:
        return None
    text = "\n".join(l for l in completion.split("\n") if not l.strip().startswith("```"))
    blocks = re.findall(r"```(?:python|py)?\s*\n(.*?)```", completion, re.DOTALL)
    code_blocks = [b for b in blocks if re.search(r"(?m)^\s*(?:class|def|import|from|@)\s", b)]

    cands = []
    # (B) from the FIRST top-level code construct to the end — keeps helper
    #     classes/functions defined above `class Solution` (fixes the drop bug).
    m = re.search(r"(?m)^(?:class|def|import|from|@)\s", text)
    if m:
        cands.append(text[m.start():])
    # (A) all fenced code blocks concatenated — helper and Solution in separate blocks.
    if code_blocks:
        cands.append("\n\n".join(code_blocks))
    # (C) a single block that names the target (old high-confidence path).
    for b in blocks:
        if "class Solution" in b or f"def {entry}" in b:
            cands.append(b)
    # (D) old anchor slice, last resort.
    for anchor in ("class Solution", f"def {entry}"):
        i = text.find(anchor)
        if i != -1:
            cands.append(text[i:])

    for c in cands:
        t = _trim_to_compilable(c)
        if t and ("class Solution" in t or f"def {entry}" in t):
            return t
    return cands[0] if cands else None


def resolve_callable(ns, entry):
    """Find the `entry` implementation regardless of how the model SHAPED its code.

    LCB expects `Solution().<entry>(*args)`. But a correct solution is sometimes
    delivered in a different container — a bare module-level function, a method on a
    RENAMED top-level class, or a method on a class nested one level inside another.
    Those are code *smells*, not wrong answers, yet the old resolver (hardcoded
    `ns['Solution']`) scored them FAIL. We look past the shape and bind to the actual
    implementation. Returns a callable (bound, if a method) or None.

    Order: (1) a bare module-level function named `entry`; else (2) a class exposing
    a callable `entry`, preferring one literally named `Solution`, searching both
    top-level classes and classes nested one level inside another class."""
    import inspect
    f = ns.get(entry)
    if callable(f) and not inspect.isclass(f):
        return f                                   # bare function shape
    classes = [v for v in ns.values() if inspect.isclass(v)]
    nested = [v for c in classes for v in vars(c).values() if inspect.isclass(v)]
    cands = [c for c in (classes + nested) if callable(getattr(c, entry, None))]
    if not cands:
        return None
    cands.sort(key=lambda c: c.__name__ != "Solution")   # prefer the expected name
    try:
        inst = cands[0]()
    except Exception:
        return None
    return getattr(inst, entry, None)


def eq(a, b):
    if isinstance(a, float) or isinstance(b, float):
        try:
            return abs(float(a) - float(b)) < 1e-6
        except Exception:
            return False
    if isinstance(a, (list, tuple)) and isinstance(b, (list, tuple)):
        return len(a) == len(b) and all(eq(x, y) for x, y in zip(a, b))
    return a == b


def sample_tests(rec, k):
    pub = [] if PRIVATE_ONLY else rec.get("public_tests", [])
    priv = rec.get("private_tests", [])
    if len(pub) + len(priv) <= k:
        return pub + priv
    need = max(0, k - len(pub))
    stride = max(1, len(priv) // need) if need else 1
    return pub + (priv[::stride][:need] if need else [])


def _run_one_test(code, entry, case, timeout=6):
    """Run ONE test in a subprocess; return (status, got_repr)."""
    def worker(q):
        ns = {}
        try:
            exec(_IMPORTS + code, ns)
        except Exception as e:
            q.append(("error", f"exec failed: {type(e).__name__}: {e}")); return
        fn = resolve_callable(ns, entry)
        if fn is None:
            q.append(("error", "entry not found")); return
        try:
            got = fn(*parse_input(case["input"]))
            q.append(("pass" if eq(got, parse_lit(case["output"])) else "wrong", repr(got)))
        except Exception as e:
            q.append(("error", f"{type(e).__name__}: {e}"))
    mgr = multiprocessing.Manager(); q = mgr.list()
    p = multiprocessing.Process(target=worker, args=(q,)); p.start(); p.join(timeout)
    if p.is_alive():
        p.kill(); p.join(1); return ("timeout", f">{timeout}s (loop/TLE)")
    return q[0] if q else ("error", "worker crashed")

print("primitives + _run_one_test defined.")

## 3 · Grab the model's raw reply
This is the messy text the extractor has to turn into runnable code.

In [ ]:
r = RESP.get((TASK_ID, CONDITION, MODEL))
assert r, f"no completion for {(TASK_ID, CONDITION, MODEL)}"
entry      = r["entry_point"]
completion = r["completion"]
statement  = PROBS.get(TASK_ID, {}).get("question_content") or r.get("prompt_text", "")
print("entry_point:", entry)
print("\nPROBLEM (first 400 chars):\n", statement[:400], "...")
print("\n" + "="*72)
print("RAW MODEL REPLY:\n")
print(completion)

## 4 · Extraction, one step at a time
### 4a · Remove the ``` fence markers
Keep the lines between fences (and any prose); drop only the fence markers themselves.

In [ ]:
text = "\n".join(l for l in completion.split("\n") if not l.strip().startswith("```"))
print("text after stripping fence markers (first 600 chars):\n")
print(text[:600] + ("..." if len(text) > 600 else ""))

### 4b · Find fenced code blocks
`FENCE_RE` pulls the contents of each ```` ```...``` ```` block; we keep the ones that look like code.

In [ ]:
blocks = re.findall(FENCE_RE, completion, re.DOTALL)
code_blocks = [b for b in blocks if re.search(CODEISH_RE, b)]
print(f"{len(blocks)} fenced block(s); {len(code_blocks)} look like code")
if code_blocks:
    print("\nfirst code block (first 300 chars):\n", code_blocks[0][:300])

### 4c · Build the ordered candidate list
The extractor doesn't guess one answer — it builds several candidate programs and tries them
in order (B keeps helpers above `Solution`, A rejoins split blocks, C/D are fallbacks).

In [ ]:
cands = []
m = re.search(CONSTRUCT_RE, text)
if m:
    cands.append(("B: from first top-level construct (keeps helpers above Solution)", text[m.start():]))
if code_blocks:
    cands.append(("A: all code blocks joined", "\n\n".join(code_blocks)))
for b in blocks:
    if "class Solution" in b or f"def {entry}" in b:
        cands.append(("C: single block naming the target", b))
for anchor in ("class Solution", f"def {entry}"):
    i = text.find(anchor)
    if i != -1:
        cands.append((f"D: slice from '{anchor}'", text[i:]))
print("candidates, in the order the extractor tries them:")
for label, _ in cands:
    print("  -", label)

### 4d · Trim each candidate and pick the winner
For each candidate: `_trim_to_compilable` peels trailing prose until it parses; we accept the
**first** candidate that compiles AND still defines the target.

In [ ]:
chosen, winner = None, None
for label, c in cands:
    t = _trim_to_compilable(c)
    ok = bool(t) and ("class Solution" in t or f"def {entry}" in t)
    print(f"[{'ACCEPT' if ok else 'skip  '}] {label}")
    if ok and chosen is None:
        chosen, winner = t, label
print("\nWINNER ->", winner)
extracted = chosen

### 4e · The extracted code (and a check against the real function)

In [ ]:
print("matches score_lcb.extract_solution():", extracted == extract_solution(completion, entry))
print("\nFINAL EXTRACTED CODE:\n")
print(extracted)

### 4f · Execute it and resolve the function
Run the extracted code into a namespace, then bind the entry point whatever shape it's in.

In [ ]:
ns = {}
exec(_IMPORTS + extracted, ns)
fn = resolve_callable(ns, entry)
print("resolved callable:", fn)

## 5 · Check ONE test's input → output
This is the grading step, broken out so you can see exactly what is compared. Needs
`lcb_tests.jsonl`.

In [ ]:
rec = TESTS.get(TASK_ID)
case = None
if not rec:
    print("no lcb_tests.jsonl uploaded -> grading section skipped (upload it to run this).")
else:
    cases = sample_tests(rec, 10000)                       # all tests
    npub = 0 if PRIVATE_ONLY else len(rec.get("public_tests", []))
    case = cases[TEST_INDEX]
    tag = "public" if TEST_INDEX < npub else "private"
    print(f"TEST #{TEST_INDEX} [{tag}]  of {len(cases)} total")
    print("  raw input string :", repr(case["input"]))
    print("  raw output string:", repr(case["output"]))

### 5a · Parse the input into arguments
LCB stores the input as text, one argument per line. `parse_input` turns it into real Python
values (via `json.loads` / `ast.literal_eval` — data only, never `eval`).

In [ ]:
if case:
    args = parse_input(case["input"])
    print("parse_input ->", args)
    print("arg types  ->", [type(a).__name__ for a in args])

### 5b · Parse the expected output

In [ ]:
if case:
    expected = parse_lit(case["output"])
    print("parse_lit ->", repr(expected), "  (type:", type(expected).__name__ + ")")

### 5c · Call the solution on those arguments

In [ ]:
if case:
    got = fn(*args)
    print(f"{entry}(*args) -> {got!r}   (type: {type(got).__name__})")

### 5d · Compare — this boolean *is* the test's pass/fail
`eq` rules: floats within `1e-6`; lists/tuples compared **element-by-element** (order matters,
length matters); everything else plain `==`.

In [ ]:
if case:
    verdict = eq(got, expected)
    print("eq(got, expected) ->", verdict)
    print(f"\n  got      = {got!r}")
    print(f"  expected = {expected!r}")
    print(f"  => this test {'PASSES' if verdict else 'FAILS'}")

## 6 · The whole problem's verdict
A problem passes only if **every** test passes. Run them all and tally the outcomes.

In [ ]:
if rec:
    from collections import Counter
    tally = Counter()
    for c in cases:
        st, _ = _run_one_test(extracted, entry, c)
        tally[st] += 1
    npass = tally.get("pass", 0)
    print("per-test outcomes:", dict(tally))
    print(f"VERDICT: {npass}/{len(cases)} tests passed  ->  {'PASS' if npass == len(cases) else 'FAIL'}")
else:
    print("(no tests uploaded)")

---
**Recap of the pipeline you just traced:**
raw reply → strip fences → find/rank candidates → trim to compilable → resolve the function →
parse each test's input & expected → call → `eq` compare → all-must-pass verdict. Change
`TASK_ID` / `TEST_INDEX` at the top and re-run to trace any other problem or test.